## Pegasus Evaluation: Chain Depth and Load

Sections B.3 and B.4 of the evaluation of the paper. The evaluation is split into `2a_base_case.ipynb` (A, B.1), `2b_evidence.ipynb` (B.2, B.2b), `2c_depth_and_load.ipynb` (B.3, B.4), `2d_tails_and_percentiles.ipynb` (B.5, B.6) and `2e_iterative.ipynb` (C.1, C.2). `2a_base_case.ipynb` describes
the two data sources, the setup (A) and the metrics and methods (B.1) that every notebook uses; the parameters are
defined once in `pegasus/evaluation_setup.py`. The real data does not support either experiment yet (B.3 and B.4 say why), so this notebook runs on synthetic
data, with the synthetic base case of B.1.

In [ ]:
# Data source of every experiment:
# - "synthetic": generated services and a simulated chain of queues (pegasus/data.py, pegasus/simulation.py)
# - "real": the measured runs of the second testbed campaign in statics/results_milos_v2 (pegasus/real_data.py)
data_source = "synthetic"   # the real data does not support B.3 and B.4 yet (`skipped`), so this notebook uses synthetic data

# the parameters and helpers that every evaluation notebook shares, defined once in pegasus/evaluation_setup.py
from pegasus.evaluation_setup import *
globals().update(configure(data_source))
# the base case of B.1 that this notebook builds on: read from statics/evaluation_cache/ (written by 2a_base_case.ipynb), or
# computed here if the cache is missing or stale (b1_base)
globals().update(b1_base(data_source))
print(f"{data_source} data, {len(seeds)} seeds: chain {' -> '.join(base.path)}, per seed {n_train} training configurations per service and "
      f"{len(base.measured)} held-out configurations, {arrivals}")

## B. Coverage and Tightness of Service Bounds

The metrics, the methods and the rule *validity before tightness* are those of B.1 (`2a_base_case.ipynb`).

### B.3 Varying Chain Depth

- **Synthetic:** chains of 1 to 50 generated service types.
- **Real:** skipped, not yet adapted: the second campaign now has chains of three to six services (`b1-vision-1` to
  `b1-vision-5`), but they differ in their services, not only in their depth.
- **Baselines:** as in B.1 without DNN Sum. The conformal bound splits its budget over the services, like the union bound.

**Setting (synthetic data).** Chains of depth $|S|$ (`depths`) of $|S|$ different service types, $s_1 \to s_2 \to \dots \to s_K$
(`synthetic_chain`): the chain of depth 3 is the chain of B.1, and every deeper chain adds further service types.
Every position has its own queue, and all positions run at the same configuration. The training configurations and
the held-out configurations are those of B.1 at every depth, every service has its own training data, and the ground
truth is simulated anew for every depth. All of this is repeated for every seed in `seeds`. Metrics and methods are those of B.1.

Two things change with the depth for every method that splits the violation budget (Pegasus Node, Quantile Union,
Conformal, Oracle): each service is bounded at the level $1 - \varepsilon / |S|$, which moves further into the tail as $|S|$
grows. The runs that Quantile Union, Conformal and the Oracle read that quantile from have a fixed number of items, so
for the deepest chains only a handful of items lie beyond it and their values rest on very few observations.
Pegasus SNC does not split the budget: it bounds the whole chain at $\varepsilon$, with one parameter θ shared by all
services.

**Complexity (against the depth).** Training time and evaluation time are defined in B.1 (Metrics).
- *Training.* Every method trains one model per service, and a chain of depth $|S|$ has $|S|$ services, so every training
  time is expected to grow linearly with $|S|$.
- *Evaluation.* A node bound of Pegasus Node is one formula and a short search on it, so its cost does not depend
  on the budget $\varepsilon / |S|$ of a service, and every position of a chain gets its own node bound. Pegasus SNC adds
  one latency moment per position, on a grid of values of θ, and minimizes once over that grid. Both versions also
  predict with one GP per service. The evaluation time of both is therefore expected to grow linearly with $|S|$.
- The conclusion below the figure gives the measured exponent of $|S|$ between the two deepest chains: an exponent of 1
  is linear growth, and an exponent below 1 means that the part of the time that does not depend on $|S|$ is still
  visible at these depths.

**Figures.** Three separate figures, each with the depth $|S|$ on the horizontal axis (logarithmic). Every line is the median over the seeds, and the band around it reaches from the smallest to the largest seed; the
legend of every figure sits inside it where it hides the fewest points.
- *Tightness*, for every seed the median over its held-out configurations. The horizontal line at 1× is the measured
  quantile; in the red area below it a method is under the measured quantile and is no bound. A filled marker means
  that the method holds at that depth (it reaches the confidence level under every seed, as defined in B.1), and a
  hollow marker that it does not. Look at the markers first: only lines with filled markers are bounds whose tightness
  can be compared.
- *Training time* (s, logarithmic), one line per set of trained models; the legend names the methods that use these
  models.
- *Evaluation time* (s, logarithmic), one line per method.

In [ ]:
if "B.3" in skipped:
    print("B.3 skipped: not supported by the {} data".format(data_source))
else:
    # B.3: chains of different depth
    def compute():
        depth_truth, depth_methods, depth_seconds = {}, {}, {}   # (depth, seed) -> ...
        with warnings.catch_warnings():
            # for some services the fitted GP hyperparameters end at their bounds, and scikit-learn warns for every such model
            warnings.simplefilter("ignore", ConvergenceWarning)
            for s in seeds:
                for depth in depths:
                    data = load_chain_data(data_source, level=1 - alpha, coverage_check=coverage_check, seed=[seed, s], **depth_options(depth, s))
                    if data_source == "synthetic":   # every position a different service type, at the held-out configurations of B.1
                        assert same_configurations(data.x_test, bases[s].x_test), "the held-out configurations differ from those of B.1"
                    assert_unseen(data)
                    models = fit_services(data, data.x_train, data.y_train, model_seed=seed + s, confidence=confidence, with_dnn=False)
                    depth_methods[(depth, s)], depth_seconds[(depth, s)] = compare_methods(data, models, data.x_train, data.monitored_latencies, alpha,
                                                                                           confidence, n_calibration, model_seed=seed + s,
                                                                                           leave_out=only_base_case, x_train_load=data.x_train_load)
                    depth_truth[(depth, s)] = (data.measured, data.measured_low)   # the traces of a deep chain are large, so only the measured quantile is kept
                    if s == seeds[0] and depth == depths[-1]:   # one deepest chain, for the figure of the minimum spacing
                        deepest_path, deepest_models, deepest_x_test = data.path, models, data.x_test
                print(f"seed {s}: depths up to {depths[-1]} evaluated")
        return dict(depth_truth=depth_truth, depth_methods=depth_methods, depth_seconds=depth_seconds, deepest_path=deepest_path,
                    deepest_models=deepest_models, deepest_x_test=deepest_x_test)

    globals().update(cached("b3_depth", data_source, compute, inputs=dict(depths=depths)))

In [ ]:
if "B.3" in skipped:
    print("B.3 skipped: not supported by the {} data".format(data_source))
else:
    depth_statistics = sweep_statistics(depth_methods, depth_truth, depths, seeds, confidence)   # name -> coverage, tightness, holds, computed
    depth_times = sweep_seconds(depth_seconds, depths, seeds)                                    # name -> training, evaluation
    tightness_and_time_figure(depths, depth_statistics, depth_times, "chain depth |S| (log scale)",
                              f"B.3 chain depth, {data_source} data, {len(seeds)} seeds", p_label, name=f"b3_depth_{data_source}")

In [ ]:
if "B.3" in skipped:
    print("B.3 skipped: not supported by the {} data".format(data_source))
else:
    depth_label = lambda depth: f"depth {depth}"
    # exponent of K between the two deepest chains, from the medians over the seeds: 1 means that the time grows linearly with the depth
    growth = lambda seconds: np.log(np.median(seconds[-1]) / np.median(seconds[-2])) / np.log(depths[-1] / depths[-2])
    display(Markdown(
        f"**What this shows** ({data_source} data, ε = {alpha:g}, {len(seeds)} seeds).\n"
        + sweep_conclusion(depths, depth_statistics, sweep_competitors, depth_label) + "\n"
        + ground_truth_conclusion(depths, depth_truth, depth_statistics, seeds, depth_label, p_label) + "\n"
        + time_conclusion(depths, depth_times, sweep_competitors, depth_label)
        + f"\n- **Growth with the depth.** Between depth {depths[-2]} and depth {depths[-1]} the time grows with |S| to the power "
        f"{growth(depth_times[PROPOSED]['training']):.1f} (training of the execution-time GPs), "
        + " and ".join(f"{growth(depth_times[name]['evaluation']):.1f} (evaluation of {name})" for name in pegasus_versions) + "."
    ))

**Figure (minimum spacing).** This figure concerns Pegasus Node only; Pegasus SNC does not use the spacing between
the departures of a service. The figure below shows the deepest chain of the first seed position by position: the
processing time of every service (the mean execution time that its GP predicts, in ms) and the *minimum spacing* of its
arrivals. The minimum spacing is the smallest gap that two consecutive items can have when they arrive at the service.
It equals the processing time of the service before, because that service handles one item at a time and so cannot
release two items closer together than it needs for one. The actual gaps are usually larger. Lines are the median over
the held-out configurations, bands the first to third quartile. Where the minimum spacing lies below the processing
time, the service sits behind a faster one: items can arrive faster than it processes them for a while, and it can
queue; these positions are circled. The conclusion below the figure gives two shares over all later positions and
held-out configurations: how often the service before is faster, and how often Pegasus Node cannot use the minimum
spacing, which is more often, because Pegasus Node requires the optimistic mean of the service before to exceed the
pessimistic mean of the service itself. In those cases Pegasus Node falls back on the external arrivals.

In [ ]:
if "B.3" in skipped:
    print("B.3 skipped: not supported by the {} data".format(data_source))
else:
    # The deepest chain: processing time of every service and minimum spacing of its arrivals, as the models predict them
    processing = np.array([deepest_models["gp"][s].predict(configurations_of(deepest_x_test, s)) for s in deepest_path]) * 1e3   # (positions, configurations), ms
    spacing = processing[:-1]                                    # minimum spacing of the arrivals at the positions 2, 3, ...: processing time of the service before
    behind_faster = np.mean(spacing < processing[1:], axis=1)    # share of the held-out configurations, per position
    pessimistic_mean = np.array([mean for mean, _ in deepest_models["beliefs"]])
    no_spacing_used = np.mean(np.array(deepest_models["lower_means"])[:-1] <= pessimistic_mean[1:], axis=1)   # Pegasus needs a clear margin between the two
    positions = np.arange(1, len(deepest_path) + 1)
    quartiles = lambda values: np.quantile(values, [0.25, 0.5, 0.75], axis=1)

    from pegasus.figures import BAND_ALPHA, legend_inside, paper_style, save_figure
    print(f"B.3 minimum arrival spacing along the chain of depth {len(deepest_path)}, {data_source} data, seed {seeds[0]}, {processing.shape[1]} held-out configurations")
    with paper_style():
        fig, ax_spacing = plt.subplots(figsize=(9, 4.5))
        segments = []   # (position, low, high) of every band, so that the legend can be placed where it hides none
        for values, where, color, marker, linestyle, label in ((processing, positions, "tab:blue", "o", "-", "processing time of the service"),
                                                               (spacing, positions[1:], "tab:orange", "s", "--",
                                                                "minimum spacing of its arrivals:\nprocessing time of the service before")):
            low, middle, high = quartiles(values)
            ax_spacing.plot(where, middle, linestyle=linestyle, marker=marker, color=color, markersize=4, linewidth=1.8, label=label)
            ax_spacing.fill_between(where, low, high, color=color, alpha=BAND_ALPHA, linewidth=0)
            segments += list(zip(where, low, high))
        faster_before = quartiles(spacing)[1] < quartiles(processing[1:])[1]
        ax_spacing.scatter(positions[1:][faster_before], quartiles(processing[1:])[1][faster_before], s=70, facecolors="none", edgecolors="tab:red",
                           linewidths=1.5, zorder=3, label="service behind a faster one\n(median over the configurations)")
        ax_spacing.set_ylabel("predicted mean time (ms)")
        ax_spacing.set_xlabel(f"position in the chain of depth {len(deepest_path)}")
        ax_spacing.grid(True, linestyle=":", alpha=0.6)
        fig.tight_layout()
        handles, labels = ax_spacing.get_legend_handles_labels()
        legend_inside(ax_spacing, handles, labels, segments, max_doublings=0, fontsize=10, framealpha=0.95)
        save_figure(fig, f"b3_min_spacing_{data_source}")
        plt.show()

In [ ]:
if "B.3" in skipped:
    print("B.3 skipped: not supported by the {} data".format(data_source))
else:
    display(Markdown(
        f"**What this shows** ({data_source} data, the chain of depth {len(deepest_path)} under seed {seeds[0]}; shares over all later positions and "
        f"held-out configurations).\n"
        f"- The service before is faster in {np.mean(behind_faster):.0%} of the cases. There the minimum spacing lies below the processing time of the "
        f"service, so the service can queue even though the service before releases one item at a time.\n"
        f"- Pegasus Node cannot use the minimum spacing in {np.mean(no_spacing_used):.0%} of the cases and bounds the service with the external "
        f"arrivals there, like the first service of the chain. In the remaining {1 - np.mean(no_spacing_used):.0%} its node bound also uses the minimum spacing."
    ))

### B.4 Varying Load and Arrival Pattern

- **Synthetic:** the load (any arrival rate, B.4a) and the variability of the arrivals (renewal arrivals with
  gamma-distributed gaps of any squared coefficient of variation $C^2$, B.4b). Batching and diurnal arrivals are not
  implemented.
- **Real:** skipped (B.4a and B.4b). The campaign measured every run under Poisson arrivals at one rate that it set
  for that run, so neither the load nor the arrival pattern can be changed.
- **Baselines:** as in B.1 without DNN Sum. Quantile Union and Conformal need every service monitored at every load and
  arrival pattern; the execution-time baseline and the proposed method do not.

B.4 has two parts. B.4a changes how many items arrive per second (the load), and B.4b changes how evenly the items
arrive at the same rate (the arrival pattern).

**Setting of both parts.** For every seed, the chain, the training configurations, the per-service execution-time
models and the held-out configurations of B.1. Execution times do not depend on the arrivals, so these models are
used unchanged. Only the arrival process at the first service changes, and the ground truth is simulated anew for
every setting.

What each method knows about the arrivals: both versions of Pegasus are given the arrival process (they use the moment
generating function of the gap between two arrivals), and Allen-Cunneen is given the arrival rate and $C^2$. Quantile Union
and Conformal are trained anew for every setting, on each service monitored alone under the arrival process of that
setting. GP Sum ignores the arrivals, so its latency is the same in every setting.

#### B.4a Load

**Setting (synthetic data).** Poisson arrivals as in B.1, with the arrival rate $\lambda$ (items/s), one value of `loads` per
setting. The rate of B.1 is one of them, and that setting is the base case again. The *utilization* of a service at a
configuration is $\rho = \lambda \cdot$ (mean execution time of the service at that configuration, in seconds) ÷ (its
number of workers): the share of the time in which every worker of the service is busy. At $\rho \geq 1$ items arrive faster than the service processes them,
its queue grows without limit, and no latency quantile exists. The rates are therefore chosen so that no service is
overloaded at any held-out configuration under any seed; the code checks this with the true mean execution times of
the generated services. The *busiest service* of a chain at a configuration is the service with the largest
utilization.

**What to expect.** The waiting time at a service grows without limit as its utilization approaches 1 (in the Allen-Cunneen
formula through the factor $1 / (1 - \rho)$). With a higher rate, the queues therefore make up a larger part of the
latency; GP Sum ignores that part. Both versions of Pegasus bound a service with the pessimistic mean of its execution
time, which lies above the true mean. Where the utilization computed with the pessimistic mean reaches 1, they give no
finite bound, although the service is stable.

**Figure.** The horizontal axis is the arrival rate (items/s); under every tick stands the utilization of the busiest
service, the median over the held-out configurations of all seeds. The vertical axis is the tightness, for every seed
the median over its held-out configurations (logarithmic). The horizontal line at 1× is the measured quantile. Every
line is the median over the seeds, and the band around it reaches from the smallest to the largest seed. A filled
marker means that the method holds at that rate (it reaches the confidence level under every seed, as defined in
B.1), a hollow marker that it does not. A seed under which a method gives no finite latency at half of the held-out
configurations or more counts as infinitely loose, and the method has no point where that is the case for half of the
seeds or more. Look at the markers first, then at whether the filled lines move away from 1× toward the higher rates.

In [ ]:
if "B.4a" in skipped:
    print("B.4a skipped: not supported by the {} data".format(data_source))
else:
    # B.4a: the chains of B.1 under Poisson arrivals of different load (`loads`: arrival rate with synthetic data, utilization of the busiest service with real data)
    base_load = workload["rate"] if data_source == "synthetic" else base_utilization
    assert base_load in loads, "the load of B.1 is not one of the settings"
    # Training runs per service. Real data: more than in B.1, because close to saturation a bound only exists if the mean execution
    # time is known precisely (see the text above). Synthetic data: the models of B.1.
    load_n_train = n_pool if data_source == "real" else n_train

    def compute():
        load_methods, load_truth, load_models, load_y_train = {}, {}, {}, {}   # (load, seed) -> name -> (group, latency per held-out configuration) / (measured quantile, its lower edge); seed -> models / their training data
        with warnings.catch_warnings():
            # for some services the fitted GP hyperparameters end at their bounds, and scikit-learn warns for every such model
            warnings.simplefilter("ignore", ConvergenceWarning)
            for load in loads:
                for s in seeds:
                    data = chain_data(s, n_train=load_n_train, **{load_option: load})
                    assert same_configurations(data.x_test, bases[s].x_test), "the held-out configurations differ from those of B.1"
                    if s not in load_models:   # the execution times do not depend on the arrivals, so every load uses the same per-service models
                        load_models[s], load_y_train[s] = (base_models[s] if load_n_train == n_train else fit_services(
                            data, data.x_train, data.y_train, model_seed=seed + s, confidence=confidence, with_dnn=False)), data.y_train
                    assert all(np.array_equal(data.y_train[name], load_y_train[s][name]) for name in data.path), "the training data differs between the loads"
                    load_methods[(load, s)], _ = compare_methods(data, load_models[s], data.x_train, data.monitored_latencies, alpha, confidence, n_calibration,
                                                                 model_seed=seed + s, leave_out=only_base_case, x_train_load=data.x_train_load)
                    load_truth[(load, s)] = (data.measured, data.measured_low)
        return dict(load_methods=load_methods, load_truth=load_truth, load_models=load_models, load_y_train=load_y_train)

    globals().update(cached("b4a_load", data_source, compute, inputs=dict(loads=loads, load_option=load_option, load_n_train=load_n_train)))
    # true utilization of the busiest service per held-out configuration, the seeds one after the other; with real data it is the load itself
    load_utilization = {load: np.concatenate([busiest_utilization(base.path, bases[s].x_test, load, service_seed=s) for s in seeds])
                        if data_source == "synthetic" else np.full(len(base.measured), load) for load in loads}
    assert max(utilization.max() for utilization in load_utilization.values()) < 1, "a service is overloaded at a held-out configuration"
    print(f"{len(load_methods)} combinations of load and seed evaluated.")

In [ ]:
if "B.4a" in skipped:
    print("B.4a skipped: not supported by the {} data".format(data_source))
else:
    load_statistics = sweep_statistics(load_methods, load_truth, loads, seeds, confidence)   # name -> coverage, tightness, holds, computed
    load_axis = ("arrival rate (items/s; below: utilization of the busiest service, median over the configurations)" if data_source == "synthetic"
                 else "utilization of the busiest service of every run")
    tightness_sweep_figure(loads, load_statistics, load_axis,
                           f"B.4a load, {data_source} data: {' → '.join(base.path)}, {len(base.measured)} held-out configurations per seed,\n"
                           f"{workload['process']} arrivals, {len(seeds)} seeds", p_label,
                           tick_labels=[f"{load:g}\n{np.median(load_utilization[load]):.2f}" if data_source == "synthetic" else f"{load:g}" for load in loads],
                           log_x=False, name=f"b4a_load_{data_source}")

In [ ]:
if "B.4a" in skipped:
    print("B.4a skipped: not supported by the {} data".format(data_source))
else:
    display(Markdown(
        f"**What this shows** ({data_source} data, {workload['process']} arrivals, ε = {alpha:g}, {len(seeds)} seeds).\n"
        f"- **Load.** Utilization of the busiest service over the held-out configurations of all seeds: "
        + "; ".join(f"median {np.median(load_utilization[load]):.2f} and maximum {load_utilization[load].max():.2f} at {load_label(load)}"
                    for load in (loads[0], loads[-1])) + ".\n"
        + sweep_conclusion(loads, load_statistics, sweep_competitors, load_label)
        + f"\n- **Finite bounds.** Share of the held-out configurations at which a version of Pegasus gives a finite latency (median over the seeds): "
        + "; ".join(f"{name} " + ", ".join(f"{finite_share(load_methods, name, load, seeds):.0%} at {load_label(load)}" for load in loads)
                    for name in pegasus_versions)
        + ". Where a version gives none, the pessimistic mean execution time of a service is so large that the service counts as overloaded at that "
        f"load, although its true utilization is below 1.\n"
        + ground_truth_conclusion(loads, load_truth, load_statistics, seeds, load_label, p_label, ", for the same services and configurations")
    ))

#### B.4b Arrival Pattern

**Setting.** A *gap* is the time between two consecutive arrivals at the first service, in seconds. The gaps are
independent and gamma-distributed with the mean of B.1 (1 ÷ arrival rate) and the squared coefficient of variation
$C^2$ = variance of a gap ÷ (mean of a gap)², one value of `arrival_cv2s` per setting. $C^2 = 1$ gives the exponential
gaps of a Poisson process, as in B.1. Below 1 the arrivals are spaced more evenly; above 1 they come in bursts with
long pauses between them. The arrival rate, and with it the utilization of every service, is the same in every
setting, so only the variability of the arrivals changes.

**Figure.** The horizontal axis is $C^2$ (logarithmic), the vertical axis the tightness, for every seed the median over
its held-out configurations (logarithmic). The horizontal line at 1× is the measured quantile. Every line is the median
over the seeds, and the band around it reaches from the smallest to the largest seed. A filled marker means that the
method holds at that $C^2$ (it reaches the confidence level under every seed, as defined in B.1), a hollow marker that
it does not. Look at the markers first, then at how the filled lines move from left to right.

In [ ]:
if "B.4b" in skipped:
    print("B.4b skipped: not supported by the {} data".format(data_source))
else:
    # B.4b: the chains of B.1 under arrivals of different variability at the arrival rates of B.1
    arrival_cv2s = [0.25, 0.5, 1, 2, 4]   # squared coefficient of variation of the gaps between arrivals; 1 is the Poisson process of B.1

    def compute():
        arrival_methods, arrival_truth = {}, {}   # (cv2, seed) -> name -> (group, latency per held-out configuration) / (measured quantile, its lower edge)
        for cv2 in arrival_cv2s:
            for s in seeds:
                data = chain_data(s, arrival_process="gamma", arrival_cv2=cv2)
                assert same_configurations(data.x_test, bases[s].x_test), "the held-out configurations differ from those of B.1"
                assert np.array_equal(data.workload["rate"], bases[s].workload["rate"]), "the arrival rates differ from those of B.1"
                # the execution times do not depend on the arrivals, so the per-service models of B.1 are the models of every setting
                assert all(np.array_equal(data.y_train[name], bases[s].y_train[name]) for name in data.path), "the training data differs from that of B.1"
                arrival_methods[(cv2, s)], _ = compare_methods(data, base_models[s], data.x_train, data.monitored_latencies, alpha, confidence, n_calibration,
                                                               model_seed=seed + s, leave_out=only_base_case, x_train_load=data.x_train_load)
                arrival_truth[(cv2, s)] = (data.measured, data.measured_low)
        return dict(arrival_methods=arrival_methods, arrival_truth=arrival_truth, gamma_arrivals=arrival_text(data.workload))

    globals().update(cached("b4b_arrivals", data_source, compute, inputs=dict(arrival_cv2s=arrival_cv2s)))
    print(f"{len(arrival_methods)} combinations of arrival variability and seed evaluated.")

In [ ]:
if "B.4b" in skipped:
    print("B.4b skipped: not supported by the {} data".format(data_source))
else:
    arrival_statistics = sweep_statistics(arrival_methods, arrival_truth, arrival_cv2s, seeds, confidence)   # name -> coverage, tightness, holds, computed
    tightness_sweep_figure(arrival_cv2s, arrival_statistics, "squared coefficient of variation C² of the gaps between arrivals (log scale)",
                           f"B.4b arrival variability, {data_source} data: {' → '.join(base.path)}, {len(base.measured)} held-out configurations per seed,\n"
                           f"{gamma_arrivals}, {len(seeds)} seeds", p_label, name=f"b4b_arrivals_{data_source}")

In [12]:
if "B.4b" in skipped:
    print("B.4b skipped: not supported by the {} data".format(data_source))
else:
    cv2_label = lambda cv2: f"C² = {cv2:g}"
    display(Markdown(
        f"**What this shows** ({data_source} data, {gamma_arrivals}, ε = {alpha:g}, {len(seeds)} seeds).\n"
        + sweep_conclusion(arrival_cv2s, arrival_statistics, sweep_competitors, cv2_label) + "\n"
        + ground_truth_conclusion(arrival_cv2s, arrival_truth, arrival_statistics, seeds, cv2_label, p_label, ", at the same arrival rates")
    ))

B.4b skipped: not supported by the real data
